```{contents}
```
## Dropout Layer


Deep neural networks often **overfit**: they memorize training data rather than learning generalizable patterns.
This happens because neurons **co-adapt** — they rely on specific other neurons instead of learning robust features.

**Dropout** combats this by introducing *stochastic regularization*:

> During training, randomly disable a fraction of neurons so the network cannot rely on any single path.

This forces the network to behave like an **ensemble of many subnetworks**, improving generalization.

---

### Mathematical Perspective

Let:

* $h$ = activations of a layer
* $m \sim \text{Bernoulli}(p)$ = dropout mask
* $p$ = probability of keeping a neuron active

Training time:
$$
\tilde{h} = m \odot h
$$

Inference time:
$$
h_{\text{test}} = p \cdot h
$$
(or equivalently, scale during training and do nothing at inference — used in modern frameworks)

---

### Training vs Inference Behavior

| Phase     | What happens                              |
| --------- | ----------------------------------------- |
| Training  | Random neurons dropped                    |
| Inference | Full network used with scaled activations |
| Effect    | Implicit ensemble learning                |

---

### Workflow in Practice

1. Forward pass
2. Sample dropout mask
3. Zero-out selected activations
4. Scale surviving activations
5. Backpropagate through active neurons only
6. At inference, disable dropout

---

### Why Dropout Works

| Mechanism               | Effect                                  |
| ----------------------- | --------------------------------------- |
| Prevents co-adaptation  | Encourages independent feature learning |
| Acts as model averaging | Improves robustness                     |
| Injects noise           | Smooths loss landscape                  |

---

### Where to Use Dropout

| Layer type             | Usage                          |
| ---------------------- | ------------------------------ |
| Fully connected layers | Very common                    |
| CNN feature maps       | Use spatial dropout            |
| Transformer attention  | Use dropout in attention & FFN |
| Input layer            | Rare, but sometimes helpful    |

---

### Hyperparameter Guidelines

| Dropout rate         | Typical values |
| -------------------- | -------------- |
| Light regularization | 0.1 – 0.3      |
| Standard             | 0.3 – 0.5      |
| Heavy                | 0.5 – 0.7      |

Too high → underfitting
Too low → overfitting persists

---

### PyTorch Demonstration

```python
import torch
import torch.nn as nn
import torch.nn.functional as F

class MLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(784, 256)
        self.dropout = nn.Dropout(p=0.5)
        self.fc2 = nn.Linear(256, 10)

    def forward(self, x):
        x = F.relu(self.fc1(x))
        x = self.dropout(x)     # applied only during training
        x = self.fc2(x)
        return x
```

Usage:

```python
model = MLP()

model.train()    # dropout active
out1 = model(x)

model.eval()     # dropout disabled
out2 = model(x)
```

---

### Variants of Dropout

| Variant             | Description                                 |
| ------------------- | ------------------------------------------- |
| Standard Dropout    | Randomly drops individual neurons           |
| Spatial Dropout     | Drops entire feature maps (CNNs)            |
| DropConnect         | Drops weights instead of activations        |
| Variational Dropout | Same mask across time steps (RNNs)          |
| Monte Carlo Dropout | Dropout active at inference for uncertainty |

---

### Remediation & Best Practices

| Problem              | Remedy                      |
| -------------------- | --------------------------- |
| Model underfitting   | Lower dropout rate          |
| Training unstable    | Combine with BatchNorm      |
| Transformer training | Apply after attention & FFN |
| Small dataset        | Increase dropout            |
| Large dataset        | Reduce dropout              |

Do **not** combine very high dropout with heavy L2 regularization — it causes excessive capacity loss.

---

### Conceptual Summary

| Aspect              | Interpretation                                   |
| ------------------- | ------------------------------------------------ |
| Core idea           | Regularization via stochastic neuron suppression |
| Optimization effect | Smoother loss, flatter minima                    |
| Generalization      | Improves robustness & reduces overfitting        |
| Theoretical view    | Approximate ensemble of subnetworks              |

